In [1]:
import torch
from einops import repeat
import numpy as np
import torch.nn as nn
import torch.optim as optim
from transformers import get_cosine_schedule_with_warmup
import os
from torch.utils.data import Dataset, DataLoader
from denoisers import *

from dataset_loaders import Train_img_Dataset
# from models.NKD_linear_layer import NKD_latent_layers, NKD_latent_layers_v2, save_checkpoint, load_checkpoint
# from NKD_kenel_layer import NKD_kernel_layers_v1
# from NKD_test import Test_denoisers


/opt/anaconda3/envs/nkd-aditya/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from denoisers import NKD_denoisers
class config:
    def __init__(self):
        pass

a = NKD_denoisers.load_checkpoint('/home/aditya/NKD/output__concat_kl_51011024_22_130_30/model_step_6000.pth',
                                  'cpu')

Loaded model state from checkpoint: /home/aditya/NKD/output__concat_kl_51011024_22_130_30/model_step_6000.pth


In [3]:
def TestDenoiser(data_loader, denoisers, out_folder, device):
    # assert len(denoisers) == len(name), "Number of denoisers and names should be same"
    print('==== Test ====')
    # Create a figure to save the images
    grid = (len(denoisers), 3)

    # Loop through the validation loader
    for i, data in enumerate(data_loader):
        print(f'Image:- {i}')

        # fig, axes = plt.subplots(len(name), 3, figsize=(15, 5 * len(name)))

        clean_image, noisy_image, sigma = data  # dataset returns a tuple of (noisy_image, clean_image)
        
        # Move images to the same device as the model
        noisy_image = noisy_image.to(device=device, dtype=torch.float32).squeeze(0)
        clean_image = clean_image.to(device=device, dtype=torch.float32).squeeze(0)
        sigma = sigma.to(device=device, dtype=torch.float32)

        print(clean_image.shape)
        print(noisy_image.shape)
        psnr_noisy = calculate_psnr(clean_image, noisy_image)
        ssim_noisy = calculate_ssim(clean_image, noisy_image)
        print(f'Noisy: psnr = {psnr_noisy:.2f} | ssim = {ssim_noisy:.4f}')
        titles = []
        images = []

        for j, name in enumerate(denoisers.keys()):
            # Pass the noisy image through the model to get the denoised image
            if name=='BM3D':
                with torch.no_grad():
                    denoised_image = denoisers[name](noisy_image, sigma.squeeze().item())
                noisy_image= noisy_image.to(device)
                denoised_image = denoised_image.to(device)

            elif name == 'NKD':
                with torch.no_grad():
                    denoised_image = denoisers[name].forward_img(noisy_image, sigma)
            else:
                with torch.no_grad():
                    denoised_image = denoisers[name](noisy_image, sigma)
            
            # Calculate PSNR of denoised image
            psnr = calculate_psnr(clean_image, denoised_image)
            ssim = calculate_ssim(clean_image, denoised_image)
            # # calculate PSNR of noisy image
            print(f'{name}: psnr = {psnr:.2f} | ssim = {ssim:.4f}')
            
            images.append(noisy_image)
            titles.append(f'Noisy Image({psnr_noisy:.2f}dB)')

            images.append(denoised_image)
            titles.append(f'{name}: Denoised Image({psnr:.2f}dB)')

            images.append(clean_image)
            titles.append(f'Clean Image')

        show_tensor_image(images, grid, titles, save=True, save_path=os.path.join(out_folder, f'saved_test_{i}_{state}.png'))



In [4]:

testset = Train_img_Dataset('./data/img_train', 'gray', 30, 30, 
                             patch_size=2*(a[1].window_rad+a[1].patch_rad)+32, 
                             imgs_per_epoch=1, req_patches=True)

test_loader = DataLoader(
            testset,
            batch_size=1,  # Process one full image at a time
            shuffle=False,
            pin_memory=True
        )
    


Loading training images...
Loaded 1 images from img_house
Total loaded 1 images from: img_house
Loaded 1 images for training


In [5]:
TestDenoiser(test_loader, {'NKD': a[0]}, './', 'cpu')

==== Test ====
Image:- 0
torch.Size([1, 256, 256])
torch.Size([1, 256, 256])
Noisy: psnr = 18.62 | ssim = 0.2484
torch.Size([1, 256, 256])
torch.Size([246, 1, 11, 11])


ValueError: error in shape of input 1:torch.Size([1, 256, 256]), 2:torch.Size([246, 1])

In [6]:
def train(args, verbose = False):
    # Setup device
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

    trainset = Train_img_Dataset('./data/img_train', args.color_mode, args.sigma_min, args.sigma_max, 
                                 patch_size=args.patch_size, imgs_per_epoch=args.imgs_per_epoch, 
                                 req_patches=args.req_patches, factor=args.factor, no_patches=args.no_patches)

    train_loader = DataLoader(
                trainset,
                batch_size=1,  # Process one full image at a time
                shuffle=False,
                pin_memory=True
            )
    
    # testset = Train_img_Dataset('./data/img_train', args.color_mode, args.sigma_max, args.sigma_max, 
    #                              patch_size=2*(args.window_rad+args.patch_rad)+args.test_imgsize, 
    #                              imgs_per_epoch=1, req_patches=False)

    # test_loader = DataLoader(
    #             testset,
    #             batch_size=1,  # Process one full image at a time
    #             shuffle=False,
    #             pin_memory=True
    #         )
    
    # Create model
    in_nc = out_nc = 1 if args.color_mode == 'gray' else 3
    model = architectures[args.architecture](args, device).to(device)

    _state = f'_{str(model)}{args.sigma_min}_{args.sigma_max}'
    args.out_folder = args.out_folder+_state
    # Check if the output folder exists
    if not os.path.exists(args.out_folder):
        # Create the folder
        os.makedirs(args.out_folder)
        os.makedirs(os.path.join(args.out_folder, 'ckpt'))
        os.makedirs(os.path.join(args.out_folder, 'imgs'))
        if verbose:
            print(f"Folder created at: {args.out_folder}")

    # Setup optimizer, scheduler and criterion
    optimizer = optim.AdamW(model.parameters(), lr=args.lr)
    if args.sch == 'multistep':
        if isinstance(args.lr_step, str):
            milestones = [int(x) for x in args.lr_step.split(',')]
        else:
            milestones = [args.lr_step]
        # Multi-step learning rate scheduler
        scheduler = optim.lr_scheduler.MultiStepLR(
            optimizer, milestones=milestones, gamma=args.lr_gamma
        )
    elif args.sch == 'cosine':
        # Cosine annealing learning rate scheduler
        scheduler = optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=args.epochs
        )
    elif args.sch == 'warmup_cosine':
        scheduler = get_cosine_schedule_with_warmup(
            optimizer=optimizer,
            num_warmup_steps=args.num_warmup_steps,
            num_training_steps=args.epochs,
            num_cycles=args.num_cycles
        )
    else:
        raise ValueError(f"scehduler(={args.sch}) is not defined")
    criterion = nn.MSELoss()
    # For tracking best validation results
    best_psnr = 0
    best_ssim = 0

    # Training loop
    total_steps = 0
    model.train()

    losses = []
    val_metric = []

    # try:
    model.train()
    for epoch in range(args.epochs):
        batch_count = 0
        current_lr = optimizer.param_groups[0]['lr']

        loss = model.train_step(train_loader, criterion, optimizer, lamda=args.l1_reg_lamda)
        total_steps += 1
        batch_count += 1
        losses.append(loss)

        # Print progress
        if total_steps % args.progress_freq == 0:
            print(f"Epoch {epoch+1}/{args.epochs} | "
                    f"Loss: {loss:.6f} | lr = {current_lr}", end='\r')

        # Regular checkpoint saving
        if total_steps % args.save_freq == 0:
            if verbose:
                print('saving checkpoint')
            model.save_checkpoint(args.out_folder, {
                    'step': total_steps,
                    'epoch': epoch,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict':optimizer.state_dict(),
                    'scheduler_state_dict':scheduler.state_dict(),
                    'loss': loss,
                    'args': args,
                }, total_steps)

        # Save epoch loss to log
        with open(os.path.join(args.out_folder, 'training_log.txt'), 'a') as f:
            f.write(f'{epoch+1},{loss:.6f},{current_lr}\n')
        scheduler.step()
        
        if args.save_img and total_steps % args.save_img_freq == 0:
            TestDenoiser(test_loader, {'nkd' : model}, device, state=_state+f'_{total_steps}', out_folder=args.out_folder)


    # except KeyboardInterrupt:
    #     print("\nTraining interrupted. Saving checkpoint...")

    # except Exception as e:
    #     print(f"\nERROR: {str(e)}")
    #     raise e

    # finally:
    print("\nTraining complete!")
    print(f"Best PSNR: {best_psnr:.2f}dB")
    print(f"Best SSIM: {best_ssim:.4f}")

    # Clean up
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return model, losses, val_metric


In [7]:
if __name__ == "__main__":
    # args.seed, args.in_folder, args.out_folder, 
    # args.color_mode:  'gray' or 'rgb', args.sigma_min, args.sigma_max, args.patch_size, 
    # args.epochs, args.batch_size, args.lr, args.sch: 'multistep', 'warmup_cosine' or 'cosine', args.lr_step, args.lr_gamma, 
    # args.val_freq, args.save_freq, args.progress_freq, args.save_img_freq,
    # args.blind, args.patch_rad, args.window_rad, args.in_channel, args.latent_dim, args.latent_channel_dim
    seed = 100
    torch.manual_seed(seed)
    np.random.seed(seed)
    class config:
        def __init__(self):
            pass

    args = config()

    # MODEL PARAMETERS
    # args.architecture = 'NKD_rbf_feat_learn'
    args.architecture = 'NKD_concat_kernel_learn'
    # args.architecture = 'NKD_diff_kernel_learn'
    args.patch_rad = 5
    args.window_rad = 10
    args.in_channel = 1
    args.latent_dim = 512
    args.latent_channel_dim = 10
    args.residual_depth = 2
    args.proj_depth = 2
    args.blind = True

    # TRAINING PARAMETERS
    args.in_folder = './data'
    args.out_folder = './output'
    args.color_mode = 'gray'
    args.sigma_min = 30
    args.sigma_max = 30
    args.req_patches = True
    args.patch_size = 128
    if args.req_patches:
        args.patch_size = 2*(args.window_rad+args.patch_rad)+1
    else:
        pass
    args.epochs = 6000
    args.batch_size = 1
    args.lr = 1e-3
    args.sch = 'cosine'
    args.l1_reg_lamda = 0.00
    # #for warmup_cosine
    # args.num_warmup_steps = 200
    # args.num_cycles = 0.5
    # #for multistep
    # args.lr_step = None
    # args.lr_gamma = None

    # PROGRESS DATA PARAMETERS
    args.val_freq = 100
    args.save_freq = 200
    args.save_img = True
    args.test_imgsize = 32
    args.save_img_freq = 200
    args.progress_freq = 1

    args.imgs_per_epoch = 1
    args.factor = None
    args.no_patches = 1024

    # RESUME TRAINING ?
    args.resume_training = False
    args.checkpoint_path = ''
    verbose = False
    model, losses, val_metric = train(args, verbose)



Loading training images...
Loaded 1 images from img_house
Total loaded 1 images from: img_house
Loaded 1 images for training
Epoch 6000/6000 | Loss: 0.000343 | lr = 6.853891787450543e-119
Training complete!
Best PSNR: 0.00dB
Best SSIM: 0.0000


In [6]:
if __name__ == "__main__":
    # args.seed, args.in_folder, args.out_folder, 
    # args.color_mode:  'gray' or 'rgb', args.sigma_min, args.sigma_max, args.patch_size, 
    # args.epochs, args.batch_size, args.lr, args.sch: 'multistep', 'warmup_cosine' or 'cosine', args.lr_step, args.lr_gamma, 
    # args.val_freq, args.save_freq, args.progress_freq, args.save_img_freq,
    # args.blind, args.patch_rad, args.window_rad, args.in_channel, args.latent_dim, args.latent_channel_dim
    seed = 100
    torch.manual_seed(seed)
    np.random.seed(seed)
    class config:
        def __init__(self):
            pass

    args = config()

    # MODEL PARAMETERS
    # args.architecture = 'NKD_rbf_feat_learn'
    # args.architecture = 'NKD_concat_kernel_learn'
    args.architecture = 'NKD_diff_kernel_learn'
    args.patch_rad = 5
    args.window_rad = 10
    args.in_channel = 1
    args.latent_dim = 512
    args.latent_channel_dim = 10
    args.residual_depth = 2
    args.proj_depth = 2
    args.blind = True

    # TRAINING PARAMETERS
    args.in_folder = './data'
    args.out_folder = './output'
    args.color_mode = 'gray'
    args.sigma_min = 30
    args.sigma_max = 30
    args.req_patches = True
    args.patch_size = 128
    if args.req_patches:
        args.patch_size = 2*(args.window_rad+args.patch_rad)+1
    else:
        pass
    args.epochs = 6000
    args.batch_size = 1
    args.lr = 3e-4
    args.sch = 'cosine'
    args.l1_reg_lamda = 0.00
    # #for warmup_cosine
    # args.num_warmup_steps = 200
    # args.num_cycles = 0.5
    # #for multistep
    # args.lr_step = None
    # args.lr_gamma = None

    # PROGRESS DATA PARAMETERS
    args.val_freq = 100
    args.save_freq = 200
    args.save_img = True
    args.test_imgsize = 32
    args.save_img_freq = 200
    args.progress_freq = 1

    args.imgs_per_epoch = 1
    args.factor = None
    args.no_patches = 1024

    # RESUME TRAINING ?
    args.resume_training = False
    args.checkpoint_path = ''
    verbose = False
    model, losses, val_metric = train(args, verbose)



Loading training images...
Loaded 1 images from img_house
Total loaded 1 images from: img_house
Loaded 1 images for training
Epoch 6000/6000 | Loss: 0.000378 | lr = 2.05616753623516e-1110
Training complete!
Best PSNR: 0.00dB
Best SSIM: 0.0000


In [7]:
if __name__ == "__main__":
    # args.seed, args.in_folder, args.out_folder, 
    # args.color_mode:  'gray' or 'rgb', args.sigma_min, args.sigma_max, args.patch_size, 
    # args.epochs, args.batch_size, args.lr, args.sch: 'multistep', 'warmup_cosine' or 'cosine', args.lr_step, args.lr_gamma, 
    # args.val_freq, args.save_freq, args.progress_freq, args.save_img_freq,
    # args.blind, args.patch_rad, args.window_rad, args.in_channel, args.latent_dim, args.latent_channel_dim
    seed = 100
    torch.manual_seed(seed)
    np.random.seed(seed)
    class config:
        def __init__(self):
            pass

    args = config()

    # MODEL PARAMETERS
    # args.architecture = 'NKD_rbf_feat_learn'
    # args.architecture = 'NKD_concat_kernel_learn'
    args.architecture = 'NKD_diff_kernel_learn'
    args.patch_rad = 5
    args.window_rad = 10
    args.in_channel = 1
    args.latent_dim = 512
    args.latent_channel_dim = 10
    args.residual_depth = 2
    args.proj_depth = 2
    args.blind = True

    # TRAINING PARAMETERS
    args.in_folder = './data'
    args.out_folder = './output'
    args.color_mode = 'gray'
    args.sigma_min = 30
    args.sigma_max = 30
    args.req_patches = True
    args.patch_size = 128
    if args.req_patches:
        args.patch_size = 2*(args.window_rad+args.patch_rad)+1
    else:
        pass
    args.epochs = 6000
    args.batch_size = 1
    args.lr = 3e-4
    args.sch = 'cosine'
    args.l1_reg_lamda = 0.00
    # #for warmup_cosine
    # args.num_warmup_steps = 200
    # args.num_cycles = 0.5
    # #for multistep
    # args.lr_step = None
    # args.lr_gamma = None

    # PROGRESS DATA PARAMETERS
    args.val_freq = 100
    args.save_freq = 200
    args.save_img = True
    args.test_imgsize = 32
    args.save_img_freq = 200
    args.progress_freq = 1

    args.imgs_per_epoch = 1
    args.factor = None
    args.no_patches = 1024

    # RESUME TRAINING ?
    args.resume_training = False
    args.checkpoint_path = ''
    verbose = False
    model, losses, val_metric = train(args, verbose)



Loading training images...
Loaded 1 images from img_house
Total loaded 1 images from: img_house
Loaded 1 images for training
Epoch 6000/6000 | Loss: 0.000372 | lr = 2.05616753623516e-1110
Training complete!
Best PSNR: 0.00dB
Best SSIM: 0.0000
